In [1]:
# ================================================================
# QWEN 2.5 3B INSTRUCT — LOAD MODEL
# ================================================================

import torch
import gc
import os
import time
import pandas as pd
import numpy as np

from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

print("=" * 70)
print("LOADING QWEN 2.5 3B INSTRUCT")
print("=" * 70)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

model.eval()

print("Model loaded.")
print("Device:", model.device)

if torch.cuda.is_available():
    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

LOADING QWEN 2.5 3B INSTRUCT


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded.
Device: cuda:0
GPU: Tesla T4


In [2]:
# ================================================================
# QWEN — EXACT SAME FEW-SHOT EXAMPLES AS PHI
# ================================================================

FEW_SHOT_EXAMPLES = [
    {
        "original":
        "The spacecraft consists of two main elements: the NASA Cassini orbiter and the ESA Huygens probe.",

        "simplified":
        "The spacecraft has two main parts: the Cassini orbiter and the Huygens probe."
    },

    {
        "original":
        "Jeddah is the principal gateway to Mecca, Islam's holiest city.",

        "simplified":
        "Jeddah is the main entrance to Mecca, Islam's holiest city."
    },

    {
        "original":
        "His next work follows an especially eventful day in the life of a successful neurosurgeon.",

        "simplified":
        "His next work is about a very busy day in the life of a successful brain surgeon."
    }
]


def few_shot_prompt(sentence):

    examples_text = ""

    for example in FEW_SHOT_EXAMPLES:

        examples_text += (
            f"Original: {example['original']}\n"
            f"Simplified: {example['simplified']}\n\n"
        )

    return f"""Simplify the following sentences using clear and easy-to-understand language.

Preserve the original meaning.
Do not add explanations or commentary.
Return only the simplified sentence.

Here are some examples:

{examples_text}

Now simplify this sentence:

Original: {sentence}
Simplified:"""


print("✅ Few-shot prompt ready.")
print("Number of examples:", len(FEW_SHOT_EXAMPLES))

✅ Few-shot prompt ready.
Number of examples: 3


In [3]:
# ================================================================
# QWEN — SAMPLE 0 TEST
# ================================================================

example = asset_df.iloc[0]

source = example["original"]

prompt = few_shot_prompt(source)

messages = [
    {
        "role": "user",
        "content": prompt
    }
]

if torch.cuda.is_available():

    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()


# ================================================================
# QWEN CHAT TEMPLATE
# ================================================================

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt"
).to(model.device)


# ================================================================
# GENERATION
# ================================================================

start_time = time.perf_counter()

with torch.no_grad():

    output_ids = model.generate(
        **inputs,
        max_new_tokens=128,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

end_time = time.perf_counter()

inference_time = end_time - start_time


# ================================================================
# DECODE ONLY GENERATED TOKENS
# ================================================================

generated_ids = output_ids[
    0,
    inputs["input_ids"].shape[-1]:
]

prediction = tokenizer.decode(
    generated_ids,
    skip_special_tokens=True
).strip()


# ================================================================
# PEAK VRAM
# ================================================================

peak_vram = (
    torch.cuda.max_memory_allocated()
    / (1024 ** 3)
    if torch.cuda.is_available()
    else None
)


# ================================================================
# DISPLAY
# ================================================================

print("=" * 80)
print("QWEN 2.5 3B — FEW-SHOT SAMPLE 0")
print("=" * 80)

print("\nORIGINAL:")
print(source)

print("\nQWEN GENERATED:")
print(prediction)

print("\nInference:", round(inference_time, 3), "sec")
print("Peak VRAM:", round(peak_vram, 3), "GB")

NameError: name 'asset_df' is not defined

In [4]:
# ================================================================
# LOAD ASSET — 2359 SAMPLES
# ================================================================

from datasets import load_dataset
import pandas as pd

print("=" * 70)
print("LOADING ASSET")
print("=" * 70)

asset = load_dataset("facebook/asset")

asset_df = pd.concat(
    [
        pd.DataFrame(asset["validation"]),
        pd.DataFrame(asset["test"])
    ],
    ignore_index=True
)

print("\n" + "=" * 70)
print("✅ ASSET READY")
print("=" * 70)

print("Validation :", len(asset["validation"]))
print("Test       :", len(asset["test"]))
print("Total      :", len(asset_df))
print("Columns    :", asset_df.columns.tolist())

LOADING ASSET


README.md:   0%|          | 0.00/11.9k [00:00<?, ?B/s]

simplification/validation-00000-of-00001(…): reconstructing file:   0%|          |  0.00B /  885kB            

simplification/validation-00000-of-00001(…): downloading bytes:           |  0.00B            

simplification/test-00000-of-00001.parqu(…): reconstructing file:   0%|          |  0.00B /  170kB            

simplification/test-00000-of-00001.parqu(…): downloading bytes:           |  0.00B            

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/359 [00:00<?, ? examples/s]


✅ ASSET READY
Validation : 2000
Test       : 359
Total      : 2359
Columns    : ['original', 'simplifications']


In [5]:
# ================================================================
# QWEN — FEW-SHOT SAMPLE 0 TEST
# ================================================================

example = asset_df.iloc[0]

source = example["original"]

prompt = few_shot_prompt(source)

messages = [
    {
        "role": "user",
        "content": prompt
    }
]

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt"
).to(model.device)

start_time = time.perf_counter()

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=128,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

end_time = time.perf_counter()

inference_time = end_time - start_time

generated_ids = output_ids[
    0,
    inputs["input_ids"].shape[-1]:
]

prediction = tokenizer.decode(
    generated_ids,
    skip_special_tokens=True
).strip()

peak_vram = (
    torch.cuda.max_memory_allocated() / (1024 ** 3)
    if torch.cuda.is_available()
    else None
)

print("=" * 80)
print("QWEN 2.5 3B — FEW-SHOT SAMPLE 0")
print("=" * 80)

print("\nORIGINAL:")
print(source)

print("\nQWEN GENERATED:")
print(prediction)

print("\nInference:", round(inference_time, 3), "sec")
print("Peak VRAM:", round(peak_vram, 3), "GB")

QWEN 2.5 3B — FEW-SHOT SAMPLE 0

ORIGINAL:
Adjacent counties are Marin (to the south), Mendocino (to the north), Lake (northeast), Napa (to the east), and Solano and Contra Costa (to the southeast).

QWEN GENERATED:
Adjacent counties are Marin (south), Mendocino (north), Lake (northeast), Napa (east), and Solano and Contra Costa (southeast).

Inference: 6.492 sec
Peak VRAM: 5.786 GB


In [6]:
# ================================================================
# QWEN — SAMPLE 0 METRICS
# ================================================================

references = example["simplifications"]

scores = evaluate_one_sample(
    source,
    prediction,
    references
)

print("=" * 80)
print("QWEN 2.5 3B — FEW-SHOT SAMPLE 0")
print("=" * 80)

print("\nORIGINAL:")
print(source)

print("\nGENERATED:")
print(prediction)

print("\nMETRICS")
print("-" * 50)

print(f"SARI          : {scores['sari']:.4f}")
print(f"BLEU          : {scores['bleu']:.4f}")
print(f"ROUGE-L       : {scores['rougeL']:.4f}")
print(f"BERTScore F1  : {scores['bertscore_f1']:.4f}")
print(f"FKGL          : {scores['fkgl']:.4f}")

print("\nRESOURCE")
print("-" * 50)

print(f"Inference     : {inference_time:.3f} sec")
print(f"Peak VRAM     : {peak_vram:.3f} GB")

print("\nReferences used:", len(references))

NameError: name 'evaluate_one_sample' is not defined

In [8]:
# ================================================================
# INSTALL MISSING METRIC DEPENDENCY
# ================================================================

!pip install -q rouge_score

  Preparing metadata (setup.py) ... done


In [9]:
# ================================================================
# LOAD EVALUATION METRICS
# ================================================================

import evaluate
import textstat
import numpy as np

print("=" * 70)
print("LOADING EVALUATION METRICS")
print("=" * 70)

sari_metric = evaluate.load("sari")
bleu_metric = evaluate.load("bleu")
rouge_metric = evaluate.load("rouge")
bertscore_metric = evaluate.load("bertscore")

print("✅ All metrics loaded successfully.")

LOADING EVALUATION METRICS


✅ All metrics loaded successfully.


In [10]:
# ================================================================
# QWEN — METRIC FUNCTION
# SAME PIPELINE AS PHI
# ================================================================

def evaluate_one_sample(source, prediction, references):

    if prediction is None or str(prediction).strip() == "":
        return {
            "sari": 0.0,
            "bleu": 0.0,
            "rougeL": 0.0,
            "bertscore_f1": 0.0,
            "fkgl": 0.0
        }

    prediction = str(prediction).strip()
    references = list(references)

    # SARI
    sari_result = sari_metric.compute(
        sources=[source],
        predictions=[prediction],
        references=[references]
    )

    # BLEU
    bleu_result = bleu_metric.compute(
        predictions=[prediction],
        references=[references]
    )

    # ROUGE-L
    rouge_result = rouge_metric.compute(
        predictions=[prediction],
        references=[references]
    )

    # BERTScore — all references
    bert_predictions = [prediction] * len(references)

    bert_result = bertscore_metric.compute(
        predictions=bert_predictions,
        references=references,
        lang="en"
    )

    bertscore_f1 = float(
        np.mean(bert_result["f1"])
    )

    # FKGL
    try:
        fkgl = textstat.flesch_kincaid_grade(
            prediction
        )
    except Exception:
        fkgl = 0.0

    return {
        "sari": float(sari_result["sari"]),
        "bleu": float(bleu_result["bleu"]),
        "rougeL": float(rouge_result["rougeL"]),
        "bertscore_f1": bertscore_f1,
        "fkgl": float(fkgl)
    }


print("✅ evaluate_one_sample() is ready.")

✅ evaluate_one_sample() is ready.


In [11]:
# ================================================================
# QWEN — SCORE SAMPLE 0
# ================================================================

references = example["simplifications"]

scores = evaluate_one_sample(
    source,
    prediction,
    references
)

print("=" * 80)
print("QWEN 2.5 3B — FEW-SHOT SAMPLE 0")
print("=" * 80)

print("\nORIGINAL:")
print(source)

print("\nGENERATED:")
print(prediction)

print("\nMETRICS")
print("-" * 50)

print(f"SARI          : {scores['sari']:.4f}")
print(f"BLEU          : {scores['bleu']:.4f}")
print(f"ROUGE-L       : {scores['rougeL']:.4f}")
print(f"BERTScore F1  : {scores['bertscore_f1']:.4f}")
print(f"FKGL          : {scores['fkgl']:.4f}")

print("\nRESOURCE")
print("-" * 50)

print(f"Inference     : {inference_time:.3f} sec")
print(f"Peak VRAM     : {peak_vram:.3f} GB")

print("\nReferences used:", len(references))

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.42GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


QWEN 2.5 3B — FEW-SHOT SAMPLE 0

ORIGINAL:
Adjacent counties are Marin (to the south), Mendocino (to the north), Lake (northeast), Napa (to the east), and Solano and Contra Costa (to the southeast).

GENERATED:
Adjacent counties are Marin (south), Mendocino (north), Lake (northeast), Napa (east), and Solano and Contra Costa (southeast).

METRICS
--------------------------------------------------
SARI          : 52.8006
BLEU          : 1.0000
ROUGE-L       : 0.9143
BERTScore F1  : 0.9351
FKGL          : 12.5576

RESOURCE
--------------------------------------------------
Inference     : 6.492 sec
Peak VRAM     : 5.786 GB

References used: 10


In [12]:
# ================================================================
# QWEN 2.5 3B INSTRUCT — FEW-SHOT FULL ASSET EVALUATION
# ================================================================
# 2359 samples
# Same 3 few-shot examples
# Same prompt
# Same metrics
# Same 10 references
# Checkpoint every 25
# Resume automatically
# ================================================================

FEW_SHOT_OUTPUT = "QWEN_FEW_SHOT_ASSET_2359_FINAL.csv"
CHECKPOINT_EVERY = 25

TOTAL_SAMPLES = len(asset_df)

print("=" * 90)
print("🚀 QWEN 2.5 3B — FEW-SHOT FULL EVALUATION")
print("=" * 90)
print(f"Total samples : {TOTAL_SAMPLES}")
print(f"Output file   : {FEW_SHOT_OUTPUT}")
print(f"Checkpoint    : Every {CHECKPOINT_EVERY} samples")
print("=" * 90)


# ================================================================
# RESUME / NEW FILE
# ================================================================

if os.path.exists(FEW_SHOT_OUTPUT):

    fewshot_df = pd.read_csv(
        FEW_SHOT_OUTPUT
    )

    completed_ids = set(
        fewshot_df["sample_id"].astype(int)
    )

    print(
        f"\n🔄 Existing CSV found."
    )

    print(
        f"Already completed: "
        f"{len(completed_ids)}/{TOTAL_SAMPLES}"
    )

else:

    fewshot_df = pd.DataFrame()

    completed_ids = set()

    print("\n🆕 Starting from sample 1.")


# ================================================================
# MAIN LOOP
# ================================================================

for i in range(TOTAL_SAMPLES):

    # ------------------------------------------------------------
    # SKIP COMPLETED SAMPLES
    # ------------------------------------------------------------

    if i in completed_ids:
        continue


    example = asset_df.iloc[i]

    source = example["original"]

    references = example["simplifications"]


    # ============================================================
    # BUILD EXACT SAME FEW-SHOT PROMPT
    # ============================================================

    prompt = few_shot_prompt(source)

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]


    # ============================================================
    # CLEAR GPU MEMORY
    # ============================================================

    if torch.cuda.is_available():

        torch.cuda.empty_cache()

        torch.cuda.reset_peak_memory_stats()


    # ============================================================
    # TOKENIZATION
    # ============================================================

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    ).to(model.device)


    # ============================================================
    # GENERATION
    # ============================================================

    start_time = time.perf_counter()

    try:

        with torch.no_grad():

            output_ids = model.generate(
                **inputs,
                max_new_tokens=128,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )


        # --------------------------------------------------------
        # ONLY GENERATED TOKENS
        # --------------------------------------------------------

        generated_ids = output_ids[
            0,
            inputs["input_ids"].shape[-1]:
        ]

        prediction = tokenizer.decode(
            generated_ids,
            skip_special_tokens=True
        ).strip()


    except Exception as e:

        print(
            f"\n❌ Generation error on sample {i}: {e}"
        )

        prediction = ""


    end_time = time.perf_counter()

    inference_time = (
        end_time - start_time
    )


    # ============================================================
    # PEAK VRAM
    # ============================================================

    peak_vram = (

        torch.cuda.max_memory_allocated()
        / (1024 ** 3)

        if torch.cuda.is_available()

        else None
    )


    # ============================================================
    # METRICS
    # ============================================================

    try:

        scores = evaluate_one_sample(
            source,
            prediction,
            references
        )

    except Exception as e:

        print(
            f"\n⚠️ Metric error on sample {i}: {e}"
        )

        scores = {
            "sari": 0.0,
            "bleu": 0.0,
            "rougeL": 0.0,
            "bertscore_f1": 0.0,
            "fkgl": 0.0
        }


    # ============================================================
    # DISPLAY
    # ============================================================

    print("\n")
    print("=" * 100)

    print(
        f"🧪 SAMPLE {i + 1}/{TOTAL_SAMPLES}"
    )

    print("=" * 100)

    print("\n🔵 ORIGINAL:")
    print(source)

    print("\n🟢 QWEN GENERATED:")
    print(
        prediction
        if prediction
        else "[EMPTY]"
    )

    print("\n📊 METRICS")
    print("-" * 60)

    print(
        f"SARI          : "
        f"{scores['sari']:.4f}"
    )

    print(
        f"BLEU          : "
        f"{scores['bleu']:.4f}"
    )

    print(
        f"ROUGE-L       : "
        f"{scores['rougeL']:.4f}"
    )

    print(
        f"BERTScore F1  : "
        f"{scores['bertscore_f1']:.4f}"
    )

    print(
        f"FKGL          : "
        f"{scores['fkgl']:.4f}"
    )

    print("\n⚡ PERFORMANCE")
    print("-" * 60)

    print(
        f"Inference     : "
        f"{inference_time:.3f} sec"
    )

    print(
        f"Peak VRAM     : "
        f"{peak_vram:.3f} GB"
    )


    # ============================================================
    # SAVE RESULT
    # ============================================================

    result = {

        "sample_id": i,

        "model":
        "Qwen2.5-3B-Instruct",

        "prompting":
        "few-shot",

        "original":
        source,

        "generated":
        prediction,

        "references":
        " ||| ".join(references),

        "sari":
        scores["sari"],

        "bleu":
        scores["bleu"],

        "rougeL":
        scores["rougeL"],

        "bertscore_f1":
        scores["bertscore_f1"],

        "fkgl":
        scores["fkgl"],

        "inference_time_sec":
        inference_time,

        "peak_vram_gb":
        peak_vram
    }


    fewshot_df = pd.concat(
        [
            fewshot_df,
            pd.DataFrame([result])
        ],
        ignore_index=True
    )


    completed_ids.add(i)


    # ============================================================
    # CHECKPOINT EVERY 25
    # ============================================================

    if (
        len(completed_ids) % CHECKPOINT_EVERY == 0
        or i == TOTAL_SAMPLES - 1
    ):

        fewshot_df.to_csv(
            FEW_SHOT_OUTPUT,
            index=False
        )

        print("\n" + "=" * 70)

        print(
            f"💾 CHECKPOINT SAVED"
        )

        print(
            f"Completed: "
            f"{len(completed_ids)}/{TOTAL_SAMPLES}"
        )

        print(
            f"File: {FEW_SHOT_OUTPUT}"
        )

        print("=" * 70)


    # ============================================================
    # MEMORY CLEANUP
    # ============================================================

    del inputs
    del output_ids
    del generated_ids

    if i % 10 == 0:

        gc.collect()

        if torch.cuda.is_available():

            torch.cuda.empty_cache()


# ================================================================
# FINAL SAVE
# ================================================================

fewshot_df.to_csv(
    FEW_SHOT_OUTPUT,
    index=False
)


# ================================================================
# FINAL SUMMARY
# ================================================================

print("\n")
print("=" * 100)
print("✅ QWEN 2.5 3B FEW-SHOT COMPLETE")
print("=" * 100)

print(
    f"Completed: "
    f"{len(fewshot_df)}/{TOTAL_SAMPLES}"
)

print(
    f"CSV: {FEW_SHOT_OUTPUT}"
)

print("\n📊 OVERALL MEAN METRICS")
print("-" * 60)

print(
    f"SARI          : "
    f"{fewshot_df['sari'].mean():.4f}"
)

print(
    f"BLEU          : "
    f"{fewshot_df['bleu'].mean():.4f}"
)

print(
    f"ROUGE-L       : "
    f"{fewshot_df['rougeL'].mean():.4f}"
)

print(
    f"BERTScore F1  : "
    f"{fewshot_df['bertscore_f1'].mean():.4f}"
)

print(
    f"FKGL          : "
    f"{fewshot_df['fkgl'].mean():.4f}"
)

print("\n⚡ PERFORMANCE")
print("-" * 60)

print(
    f"Mean inference : "
    f"{fewshot_df['inference_time_sec'].mean():.4f} sec"
)

print(
    f"Mean peak VRAM : "
    f"{fewshot_df['peak_vram_gb'].mean():.4f} GB"
)

print("=" * 100)

Streaming output truncated to the last 5000 lines.

⚡ PERFORMANCE
------------------------------------------------------------
Inference     : 1.103 sec
Peak VRAM     : 6.780 GB


🧪 SAMPLE 2155/2359

🔵 ORIGINAL:
SummerSlam (2009) is an upcoming professional wrestling pay-per-view event produced by World Wrestling Entertainment (WWE), which will take place on August 23, 2009 at Staples Center in Los Angeles, California.

🟢 QWEN GENERATED:
SummerSlam (2009) is a coming professional wrestling pay-per-view event produced by WWE, happening on August 23, 2009, at Staples Center in Los Angeles, California.

📊 METRICS
------------------------------------------------------------
SARI          : 39.8375
BLEU          : 0.7369
ROUGE-L       : 0.7500
BERTScore F1  : 0.9415
FKGL          : 15.4033

⚡ PERFORMANCE
------------------------------------------------------------
Inference     : 2.326 sec
Peak VRAM     : 6.783 GB


🧪 SAMPLE 2156/2359

🔵 ORIGINAL:
Usually portrayed as being bald, with long 

In [13]:
# ================================================================
# QWEN 2.5 3B — FEW-SHOT DESCRIPTIVE STATISTICS
# ================================================================

import pandas as pd
import numpy as np

QWEN_FILE = "QWEN_FEW_SHOT_ASSET_2359_FINAL.csv"

df = pd.read_csv(QWEN_FILE)

metrics = [
    "sari",
    "bleu",
    "rougeL",
    "bertscore_f1",
    "fkgl",
    "inference_time_sec",
    "peak_vram_gb"
]

stats = []

for metric in metrics:

    stats.append({
        "metric": metric,
        "mean": df[metric].mean(),
        "median": df[metric].median(),
        "min": df[metric].min(),
        "max": df[metric].max(),
        "std": df[metric].std()
    })

stats_df = pd.DataFrame(stats)

print("=" * 100)
print("📊 QWEN 2.5 3B — FEW-SHOT DESCRIPTIVE STATISTICS")
print("=" * 100)

print(f"Total samples: {len(df)}")

print("\n")
print(
    stats_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

print("=" * 100)

📊 QWEN 2.5 3B — FEW-SHOT DESCRIPTIVE STATISTICS
Total samples: 2359


            metric      mean    median       min       max      std
              sari 45.516303 45.173608 19.368030 80.275632 8.455622
              bleu  0.724481  0.750534  0.000000  1.000000 0.215248
            rougeL  0.810460  0.823529  0.279070  1.000000 0.124863
      bertscore_f1  0.954847  0.956226  0.879700  0.993929 0.015217
              fkgl  9.400441  9.255714 -1.840000 25.434706 3.651786
inference_time_sec  1.373226  1.269397  0.393086  3.799805 0.547857
      peak_vram_gb  6.780018  6.780013  6.776754  6.788528 0.001717
